In [5]:
import random

alphabet_num = 10
length = 20 # 長さは固定
convert_length = 3

alphabet = tuple(chr(i + 65) for i in range(alphabet_num))

rule_num = 200

rules = []
while len(rules) < rule_num:
    before = ''.join(random.choices(alphabet, k=convert_length))
    after = ''.join(random.choices(alphabet, k=convert_length))
    if (before != after) and ((before, after) not in rules):
        rules.append((before, after))

print(rules)

[('BJJ', 'GCG'), ('BCH', 'GBJ'), ('CEB', 'IFB'), ('DDD', 'DIG'), ('BCI', 'JHB'), ('ADA', 'EJD'), ('IHE', 'JBJ'), ('IIB', 'CGI'), ('BGF', 'JBH'), ('EFH', 'IDA'), ('JIB', 'JJJ'), ('EIC', 'CGI'), ('EFJ', 'HEC'), ('CAG', 'IID'), ('HIG', 'JCB'), ('EIF', 'JAC'), ('CJD', 'JAG'), ('DDE', 'EEI'), ('IIE', 'JHG'), ('BEG', 'DGE'), ('EDG', 'EGA'), ('DGF', 'JBH'), ('HJE', 'FCG'), ('FFG', 'FDG'), ('GBI', 'CEB'), ('HBB', 'FHB'), ('BBD', 'JFD'), ('BJH', 'GCH'), ('AIC', 'BCB'), ('FFC', 'DJE'), ('FGA', 'EJC'), ('EFJ', 'CAH'), ('CCJ', 'GIB'), ('EFI', 'BCF'), ('HFA', 'JDC'), ('HED', 'JJF'), ('GCG', 'HBE'), ('FAF', 'DEA'), ('HAI', 'FIF'), ('BJC', 'IAI'), ('DJG', 'EDC'), ('CDB', 'FCH'), ('JDH', 'HHI'), ('IAC', 'HDA'), ('IDG', 'IHA'), ('HBG', 'GJI'), ('EFF', 'FFG'), ('JDD', 'IIC'), ('HCE', 'GBJ'), ('GEB', 'BFH'), ('JCI', 'GAI'), ('GIC', 'GDG'), ('GAC', 'HFD'), ('BEF', 'AAB'), ('AIC', 'EBB'), ('DGE', 'EEH'), ('IFI', 'BAJ'), ('JIF', 'ICD'), ('HAC', 'JCI'), ('JDA', 'GGJ'), ('BHE', 'HED'), ('BFJ', 'JHA'), ('EBJ',

In [6]:
def ngram(a, b, n):
    assert len(a) == len(b)
    return sum(1 for i in range(len(a) - n + 1) if a[i:i+n] == b[i:i+n])

def find_successors(s, rules):
    out = []
    for lhs, rhs in rules:
        k = len(lhs)
        start = 0
        while True:
            idx = s.find(lhs, start)
            if idx == -1:
                break
            nxt = s[:idx] + rhs + s[idx + k:]
            out.append((nxt, (idx, lhs, rhs)))
            start = idx + 1
    return out


In [32]:
import random

def sample_path(rules, alphabet, length=20, step_num=10, direction="forward"):
    found = False
    path = []
    while not found:
        before = ''.join(random.choices(alphabet, k=length))
        state = before
        path = [state]

        for i in range(step_num):
            successors = find_successors(state, rules)

            if len(successors) == 0:
                break
            else:
                if direction == "forward":
                    state = random.choice(successors)[0]
                elif direction == "backward":
                    state = random.choice(successors)[1]
                else:
                    raise ValueError("direction must be 'forward' or 'backward'")
                
                path.append(state)

            if i == step_num - 1:
                found = True
        
    after = state
    return before, after, path

before, after, path = sample_path(rules, alphabet, length=20, step_num=10, direction="forward")
print("before:", before)
print("after:", after)
print("path:", path)

before: DCEFFGBFEHEGBBFJJEEE
after: BDIFEBGEBGCABJFFEEEE
path: ['DCEFFGBFEHEGBBFJJEEE', 'DCEFFGBFEHEGBJHAJEEE', 'DCEFDGBFEHEGBJHAJEEE', 'DCEFDGBFBEEGBJHAJEEE', 'DCEFDGBFDIDGBJHAJEEE', 'DCEFDGBFDIDGBJFFEEEE', 'DCEFEEJFDIDGBJFFEEEE', 'DCEFEEJFDIHABJFFEEEE', 'DCEFEEJFBGCABJFFEEEE', 'BDIFEEJFBGCABJFFEEEE', 'BDIFEBGEBGCABJFFEEEE']


In [ ]:
def beam_search(before, after, rules, width=100, ngram_n=1, max_steps=50):
    state_list = [(before, ngram(before, after, ngram_n), [before])]
    visited = set([before])

    for _ in range(max_steps):
        new_state_list = []

        for state, _, path in state_list:
            successors = find_successors(state, rules)
            for succ, _ in successors:  # successor = (next_state, rule)
                if succ in visited:
                    continue
                visited.add(succ)

                score = ngram(succ, after, ngram_n)
                new_path = path + [succ]

                # ゴール判定は文字列一致
                if succ == after:
                    return new_path

                new_state_list.append((succ, score, new_path))

        if not new_state_list:
            break

        # スコアが大きい順にソート（良いものを残す）
        state_list = sorted(new_state_list, key=lambda x: x[1], reverse=True)[:width]

    return None



path = beam_search(before, after, rules, width=100, ngram_n=1)
if path:
    print("Found path:", path)
else:
    print("No path found")





Found path: ['DCEFFGBFEHEGBBFJJEEE', 'BDIFFGBFEHEGBBFJJEEE', 'BDIFFGBFBEEGBBFJJEEE', 'BDIFDGBFBEEGBBFJJEEE', 'BDIFEEJFBEEGBBFJJEEE', 'BDIFEBGEBEEGBBFJJEEE', 'BDIFEBGEDIDGBBFJJEEE', 'BDIFEBGEDIHABBFJJEEE', 'BDIFEBGEBGCABBFJJEEE', 'BDIFEBGEBGCABJHAJEEE', 'BDIFEBGEBGCABJFFEEEE']
